In [34]:
# Load env variables
import os
from dotenv import load_dotenv

load_dotenv()
os.environ.pop('ANTHROPIC_AUTH_TOKEN', None)  # workaround: unset empty env var that breaks SDK

# Create an API client
from anthropic import Anthropic

client = Anthropic()
#model = "claude-sonnet-4-6"
#model = "claude-opus-4-6"
model = "claude-haiku-4-5"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text


In [35]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate an evaluation dataset for prompt evaluation. The dataset will be used to evaluate prompt that generate Python, JSON, or regex. Specifically for Python learning tasks, generate an array of JSON, each representing tasks that require Python, JSON, or regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
        "solution_criteria": "Must include topics like 'Python syntax model', or 'Python Embedded object types' or 'Advanced Python tools, including decorators, descriptors, and metaclasses'"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single pattern function or single JSON object, or regex.
* Focus on tasks that do not require writing much code.
Please generate five objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [36]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [37]:
def run_prompt(test_case):
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}

    *  Respond only with Python, JSON, or plain regex.
    * Do not add any comments or commentary or explanation.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [38]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an proficient Python developer and experienced interviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution criteria:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [39]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text)
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [40]:
def run_test_case(test_case):
    """Calls the run and the score prompt, then grades the result."""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade.get("score")
    reasoning = model_grade.get("reasoning")

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [41]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [42]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

    results = run_eval(dataset)

Average score: 8.3


In [43]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n  \"database\": {\n    \"host\": \"localhost\",\n    \"port\": 5432,\n    \"username\": \"app_user\",\n    \"password\": \"secure_password_123\",\n    \"database_name\": \"web_app_db\",\n    \"pool_size\": 10,\n    \"timeout\": 30\n  },\n  \"api_endpoints\": {\n    \"base_url\": \"https://api.example.com\",\n    \"version\": \"v1\",\n    \"endpoints\": {\n      \"users\": \"/users\",\n      \"products\": \"/products\",\n      \"orders\": \"/orders\",\n      \"auth\": \"/auth\"\n    },\n    \"timeout\": 15,\n    \"retry_attempts\": 3\n  },\n  \"logging\": {\n    \"level\": \"INFO\",\n    \"format\": \"%(asctime)s - %(name)s - %(levelname)s - %(message)s\",\n    \"handlers\": {\n      \"console\": {\n        \"enabled\": true,\n        \"level\": \"DEBUG\"\n      },\n      \"file\": {\n        \"enabled\": true,\n        \"level\": \"INFO\",\n        \"path\": \"/var/log/app.log\",\n        \"max_size\": \"10MB\",\n        \"backup_count\": 5\n      }\n    }\n  